# Bias Fitting with the Analytic Covariance

Same as `Bias_Fitting.ipynb`, but the sample (leave-one-out) covariances are replaced by the analytic ones from
`Analytic_Covariance.ipynb` (run that first, with the same `shift_kmax`, for the chosen `cov_version`).

* Power spectrum: the 127-bin analytic covariance is rebinned to the fitting bins with the same weights used to bin the
  data, times the window $W(k)^2$ the templates carry: $C_\mathrm{bin} = M C M^T$ with $M_{bi} = W(k_i)^2 N_i / N_b$.
  With `bin_P = False` every measured k-bin up to `kMax_P` is its own fitting bin, so $M$ only applies $W(k)^2$.
* Bispectrum: the full analytic covariance (off-diagonal terms included), times $W_{123}W_{123}'$.
* P–B cross covariance: rebinned the same way on the P side, times $W_{123}$ on the B side.
* P and B are fit jointly with the full covariance
  $\begin{pmatrix} C_{PP} & C_{PB} \\ C_{PB}^T & C_{BB} \end{pmatrix}$, so
  $\chi^2 = \chi^2_P + \chi^2_B + \chi^2_{PB}$, where $\chi^2_P$ and $\chi^2_B$ are the $PP$ and $BB$ blocks of
  $r^T C^{-1} r$ and $\chi^2_{PB}$ is the cross term (twice the $PB$ block).
* The covariance is the same for every phase (no leave-one-out, so no Hartlap factor).


In [1]:
import sys
sys.path.append('/home/kboone/software/Galaxy_Bias/Bispectrum/Sim_BiasFitting/')

import os
import jax
import h5py
import numpy as np
import jax.numpy as jnp
from Fits_Helpers import *
from scipy.optimize import least_squares
from scipy.interpolate import CubicSpline as spline

jax.config.update("jax_enable_x64", True)

In [2]:
cov_version = 'v2'  # 'v1' or 'v2', see Analytic_Covariance.ipynb
no_cross = False # If true, no cross covariance
bin_P = False # If false, every measured power spectrum k-bin up to kMax_P is fit individually (no rebinning)

if no_cross:
    cov_dir = 'New_Cov/'
    if not bin_P:
        cov_dir = 'New_Cov_no_Bins/'
else:
    cov_dir = 'New_Cov_w_Cross/'
    if not bin_P:
        cov_dir = 'New_Cov_w_Cross_no_Bins/'

z = 8
shift_kmax = 1.0  # Must match the shift_kmax used in Analytic_Covariance.ipynb
pos_root = '../../Data/Gala_z8_Positions/'
sim_root = '/mnt/nvme1/kboone/Data/BigFiles/Gala_8/'
model_root = '/mnt/nvme1/kboone/Sims/'
cov_root = 'Covariances/'
assert cov_version in ('v1', 'v2'), "cov_version must be 'v1' or 'v2'"

rescale = (8+1)/(z+1)
L = 2000
kF = 2*np.pi/L
kN_tsc = 2048*np.pi/L
kN = 256*np.pi/L
phases = 25

In [3]:
noises = np.zeros(phases, dtype=np.float64)
for ph in np.arange(phases):
    with h5py.File(pos_root + 'Gala_ph0' + '{:02d}'.format(ph) + '.h5', 'r') as f_:
        noises[ph] = L**3/f_['dataset'].shape[1]

In [4]:
def W(k):
    return (np.sin(np.pi*k/(2*kN_tsc)) / (np.pi*k/(2*kN_tsc)))**3

In [5]:
# Power spectrum data
numBins = 12
kMax_P = kN * shift_kmax

kBins = np.geomspace(kF, kMax_P, numBins + 1)
kBins = np.maximum(kBins, (np.arange(numBins+1)+0.5)*kF)
if not bin_P:
    # One fitting bin per measured k-bin in the same k range: edges halfway between neighbouring k values.
    k_fine = data_power(sim_root + 'Gala_ph000/Gala_256_ps.dat', kBins, numBins)[1]
    k_fine = k_fine[(k_fine >= kBins[0]) & (k_fine < kBins[-1])]
    kBins = np.concatenate([[kBins[0]], 0.5*(k_fine[1:] + k_fine[:-1]), [kBins[-1]]])
    numBins = len(k_fine)
P_bins = []
for ph in np.arange(phases):
    power_file = sim_root + 'Gala_ph0' + '{:02d}'.format(ph) + '/Gala_256_ps.dat'
    P_bins_ind, kP, bin_idx, mask_P, N_P, N_P_tot = data_power(power_file, kBins, numBins)
    P_bins.append(P_bins_ind)
P_bins = np.array(P_bins)
std_P_bins = np.std(P_bins, axis=0, ddof=1)
Wk = W(kP)

# When multiplied by noises[ind] these will give the constant and k^2 power spectrum noise contributions.
Pnoise_c  = np.bincount(bin_idx[mask_P], weights=(Wk**2*N_P)[mask_P], minlength=numBins)/N_P_tot
Pnoise_k2 = np.bincount(bin_idx[mask_P], weights=(Wk**2*kP**2*N_P)[mask_P], minlength=numBins)/N_P_tot

In [6]:
# Bispectrum data
max_ksum = kN * shift_kmax
min_ksum = max_ksum/4

min_k = 0.5*kF * shift_kmax
max_k = 64.5*kF * shift_kmax

Bs = []
for ph in np.arange(phases):
    bs_file = sim_root + 'Gala_ph0' + '{:02d}'.format(ph) + '/Gala_256_comb.dat'
    B, kB = data_bispec(bs_file, min_k, max_k, min_ksum, max_ksum)
    Bs.append(B)
Bs = np.array(Bs)
std_B = np.std(Bs, axis=0, ddof=1)
k1, k2, k3 = kB[:,0], kB[:,1], kB[:,2]
W123 = W(k1)*W(k2)*W(k3)

# Number of bins for the power spectrum and bispectrum respectively.
N_Pb, N_Bb = numBins, Bs.shape[1]

In [7]:
# name: (sym_factor, rescale power)
P_files = {'b1b1': (1, 2), 'F2F2': (1, 4), 'b1F3': (2, 4), 'b2b2': (1, 4), 'b2F2': (2, 4),
           'b2K2': (2, 4), 'K2K2': (1, 4), 'K2F2': (2, 4), 'b1K2_2': (2, 4), 'b1Otd': (2, 4),
           'b1bk2': (-2, 2), 'b3b3': (1, 6)}
B_files = {'b1b1b2': (3, 4), 'b1b1K2': (3, 4), 'b1b1F2': (3, 4), 'b2b2b2': (1, 6), 'b1b2b3': (6, 6)}

P_tmpl, P_sum, B_tmpl = {}, {}, {}
for name, (sym, rpow) in P_files.items():
    unbinned, binned = sims_power(model_root + 'PowerSpectra/' + name + '_ps.dat', kBins, numBins, Wk, sym_factor=sym)
    P_tmpl[name] = binned * rescale**rpow
    S = spline(kP, unbinned * rescale**rpow, bc_type='natural')
    P_sum[name] = S(k1) + S(k2) + S(k3)
for name, (sym, rpow) in B_files.items():
    B_tmpl[name] = sims_bispec(model_root + 'Bispectra/' + name + '_comb.dat', kB, sym_factor=sym) * W123 * rescale**rpow

In [8]:
# # Laplacian sign check: bk2 multiplies the Laplacian of delta, which is -k^2 delta, so with sym_factor -2 the
# # b1bk2 template should be exactly -2 k^2 P_11 (b1b1) at every k.
# import matplotlib.pyplot as plt
# P_lap = sims_power(model_root + 'PowerSpectra/b1bk2_ps.dat', kBins, numBins, Wk, sym_factor=-2)[0]
# P_lin = sims_power(model_root + 'PowerSpectra/b1b1_ps.dat', kBins, numBins, Wk, sym_factor=1)[0]
# plt.semilogx(kP[mask_P], (P_lap/(kP**2*P_lin))[mask_P], '.', ms=2)
# plt.axhline(-2, c='k', ls='--'); plt.xlabel('k [h/Mpc]'); plt.ylabel(r'$P_{b_1 b_{k^2}} / (k^2 P_{11})$');

In [9]:
# Analytic power spectrum covariance, rebinned to the fitting bins.
# M_bin[b, i] = W(k_i)^2 N_i / N_b for the fine bins i in fitting bin b (same weights as the binned templates).
C_P_fine = np.load(cov_root + 'Power_Cov_' + cov_version + '.npy')
assert C_P_fine.shape == (len(kP), len(kP))
M_bin = np.zeros((numBins, len(kP)))
M_bin[bin_idx[mask_P], np.where(mask_P)[0]] = (Wk**2*N_P)[mask_P] / N_P_tot[bin_idx[mask_P]]
C_PP = M_bin @ C_P_fine @ M_bin.T

## Bispectrum and cross covariance (analytic), joint inverse

In [10]:
# Analytic bispectrum covariance (full) and P-B cross covariance, times the window on each triangle.
kB_cov = np.load(cov_root + 'Bispec_Cov_k.npy')
assert kB_cov.shape == kB.shape and np.allclose(kB_cov, kB), 'Covariance triangles do not match (check shift_kmax)'
C_BB = np.load(cov_root + 'Bispec_Cov_' + cov_version + '.npy') * np.outer(W123, W123)
C_PB = (M_bin @ np.load(cov_root + 'Cross_Cov.npy')) * W123[None, :]
if no_cross:
    C_PB = np.zeros_like(C_PB)

# Joint covariance of (P bins, B triangles), in units of (std_P_bins, std_B), and its inverse.
std_PB = np.concatenate([std_P_bins, std_B])
C_joint = np.block([[C_PP, C_PB], [C_PB.T, C_BB]]) / np.outer(std_PB, std_PB)
print('min eigenvalue of the joint covariance (units of std):', np.linalg.eigvalsh(C_joint).min())
C_inv_joint = np.linalg.inv(C_joint)


# R with R.T @ R = M, so that c^T M c = |R c|^2 (needed for least_squares).
def sqrt_psd(M):
    w, V = np.linalg.eigh(M)
    return np.sqrt(np.clip(w, 0, None))[:, None] * V.T

min eigenvalue of the joint covariance (units of std): 0.013625762703072133


## Model contributions

In [11]:
# Power spectrum contributions                    Bispectrum contributions
#   idx  template   bias coefficient                  idx  template   bias coefficient
#   0    b1b1       b1^2                              0    b1b1b2     b1^2 b2
#   1    F2F2       b1^2                              1    b1b1K2     b1^2 bK2
#   2    b1F3       b1^2                              2    b1b1F2     b1^3
#   3    b2b2       b2^2                              3    b2b2b2     b2^3
#   4    b2F2       b1 b2                             4    b1b2b3     b1 b2 b3
#   5    b2K2       b2 bK2                            5    constant noise   bBCN
#   6    K2K2       bK2^2
#   7    K2F2       b1 bK2                         The bispectrum shot noise bBPN * sum_P(k1)+P(k2)+P(k3)
#   8    b1K2_2     b1 bK2                         uses whichever of P contributions 0-11 are kept.
#   9    b1Otd      b1 bOtd
#   10   b1bk2      b1 bk2
#   11   b3b3       b3^2
#   12   constant noise   bPN
#   13   k^2 noise        bPk2N
P_names = ['b1b1', 'F2F2', 'b1F3', 'b2b2', 'b2F2', 'b2K2', 'K2K2', 'K2F2', 'b1K2_2', 'b1Otd', 'b1bk2', 'b3b3']
B_names = ['b1b1b2', 'b1b1K2', 'b1b1F2', 'b2b2b2', 'b1b2b3']
param_names = ['bPN', 'bPk2N', 'bBCN', 'bBPN', 'b1', 'b2', 'bK2', 'bOtd', 'bk2', 'b3']

# option: (P indexes to drop, B indexes to drop, parameters fixed to 0)
DROP = {'no_k2_noise':  ([13],            [],        ['bPk2N']),
        'no_nl_matter': ([1, 2, 4, 7, 8], [2],       []),
        'no_tidal':     ([5, 6, 7, 8],    [1],       ['bK2']),
        'no_btd':       ([9],             [],        ['bOtd']),
        'no_blap':      ([10],            [],        ['bk2']),
        'no_b3':        ([11],            [3, 4],    ['b3']), # No b3 also for me means dropping to tree level bispectrum
        'no_b2':        ([3, 4, 5],       [0, 3, 4], ['b2'])}

In [12]:
# Only change the variables in this block from run to run.
# Full: only no_b3. Full + b3: nothing. Linear matter: no_b3, no_nl_matter.
# Alt: no_nl_matter, no_tidal, no_btd, no_blap. Simple: Alt + no_k2_noise.
model = dict(no_k2_noise=False, # DO NOT CHANGE THIS FROM FALSE
             no_nl_matter=True, no_tidal=True, 
             no_btd=True, no_blap=True, no_b3=False, no_b2=False)
write_root = cov_dir + cov_version + '_Lin_b3_'
os.makedirs(os.path.dirname(write_root), exist_ok=True)

on = [DROP[key] for key, val in model.items() if val]
drop_P = set().union(*[d[0] for d in on])
drop_B = set().union(*[d[1] for d in on])
fixed = set().union(*[d[2] for d in on])

keep_P = np.array([i for i in range(14) if i not in drop_P])
keep_B = np.array([i for i in range(6) if i not in drop_B])
keep_PS = keep_P[keep_P < 12]
free = np.array([i for i, name in enumerate(param_names) if name not in fixed])
print('Free parameters:', [param_names[i] for i in free])

Free parameters: ['bPN', 'bPk2N', 'bBCN', 'bBPN', 'b1', 'b2', 'b3']


## Fitting

In [13]:
# Joint chi2 matrix for phase ind: chi2 = v^T M v with v = (P_vec, B_vec). Also returns len(P_vec).
def chi2_matrix(ind):
    P_terms = np.array([P_tmpl[name] for name in P_names]
                       + [noises[ind]*Pnoise_c,
                          noises[ind]*Pnoise_k2])[keep_P]
    P_terms = np.vstack([P_terms, -P_bins[ind]])/std_P_bins

    B_terms = np.array([B_tmpl[name] for name in B_names]
                       + [W123*noises[ind]**2])[keep_B]
    B_noise = np.array([W123*noises[ind]*P_sum[name] for name in P_names])[keep_PS]
    B_terms = np.vstack([B_terms, B_noise, -Bs[ind]])/std_B

    nP, nB = len(P_terms), len(B_terms)
    T = np.zeros((nP + nB, N_Pb + N_Bb))
    T[:nP, :N_Pb] = P_terms
    T[nP:, N_Pb:] = B_terms
    return T @ C_inv_joint @ T.T, nP


# (chi2_P, chi2_B, chi2_PB) of the joint vector v; they sum to v^T M v.
def chi2_parts(v, M, nP):
    return np.array([v[:nP] @ M[:nP, :nP] @ v[:nP], v[nP:] @ M[nP:, nP:] @ v[nP:], 2*v[:nP] @ M[:nP, nP:] @ v[nP:]])


def bias_terms(params):
    bPN, bPk2N, bBCN, bBPN, b1, b2, bK2, bOtd, bk2, b3 = params
    P_bias_terms = jnp.array([b1**2, b1**2, b1**2, b2**2, b1*b2, b2*bK2, bK2**2,
                              b1*bK2, b1*bK2, b1*bOtd, b1*bk2, b3**2, bPN, bPk2N])
    B_bias_terms = jnp.array([b1**2*b2, b1**2*bK2, b1**3, b2**3, b1*b2*b3, bBCN])
    P_vec = jnp.append(P_bias_terms[keep_P], 1.0)
    B_vec = jnp.concatenate([B_bias_terms[keep_B], bBPN*P_bias_terms[keep_PS], jnp.ones(1)])
    return jnp.concatenate([P_vec, B_vec])


def fit(ind):
    M, nP = chi2_matrix(ind)
    R = jnp.asarray(sqrt_psd(M))

    def _residuals(free_params):
        params = jnp.zeros(10).at[free].set(free_params)
        return R @ bias_terms(params)

    residuals = jax.jit(_residuals)
    jacobian = jax.jit(jax.jacfwd(_residuals))

    # Starting point depends on redshift (bias is much smaller at z = 5).
    if z == 8:
        ind_initial_guess = np.array([1.0, 1.0, 1.0, 1.0, 10.0, 40.0, 1.0, 10.0, 1.0, 100.0])
    elif z == 5:
        ind_initial_guess = np.array([1.0, 1.0, 1.0, 1.0, 5.0, 6.0, -1.0, 1.0, 1.0, 10.0])

    sol = least_squares(lambda x: np.asarray(residuals(x)), ind_initial_guess[free], 
                          jac=lambda x: np.asarray(jacobian(x)), method='trf', x_scale='jac', 
                          xtol=1e-15, ftol=1e-15, gtol=1e-15, max_nfev=5000)

    ind_best_params = np.zeros(10)
    ind_best_params[free] = sol.x
    ind_chi2 = chi2_parts(np.asarray(bias_terms(ind_best_params)), M, nP)
    return ind_best_params, ind_chi2

## Run

In [14]:
best_params = []
chi2 = []
for ind in np.arange(phases):
    best_params_ind, chi2_ind = fit(ind)
    best_params.append(best_params_ind)
    chi2.append(chi2_ind)
best_params = np.array(best_params)
chi2 = np.array(chi2)

# Best fits: all 10 parameters in param_names order (fixed ones are 0). Chi2: (chi2_P, chi2_B, chi2_PB) per phase.
np.save(write_root + 'Best_Fits.npy', best_params)
# np.save(write_root + 'Chi2.npy', chi2)

dof = N_Pb + N_Bb - len(free)
print(f'chi2_P/N_P = {chi2[:,0].mean()/N_Pb:.3f}   chi2_B/N_B = {chi2[:,1].mean()/N_Bb:.3f}   chi2_PB = {chi2[:,2].mean():.1f}   '
      f'total = {chi2.sum(1).mean():.1f}   total/dof = {chi2.sum(1).mean()/dof:.3f}')

chi2_P/N_P = 1.106   chi2_B/N_B = 1.232   chi2_PB = -34.9   total = 3049.5   total/dof = 1.215


## Cross-validation

In [15]:
# Cross-validation: score each phase's best-fit bias parameters (from the Run cell) on every phase's data.
# cv_chi2[i, j] = (chi2_P, chi2_B, chi2_PB) of phase i's best-fit bias parameters (b1, b2, ...) on phase j's data, using
# phase j's shot noise and the analytic covariance, with the free noise parameters (bPN, bPk2N, bBCN, bBPN) refit on
# phase j. P and B share one covariance, so the four noise parameters are refit jointly.
# The diagonal (i = j) is the usual training chi2; the off-diagonal entries are test chi2.
# With the bias parameters fixed, v is linear in the noise parameters, so chi2 is quadratic in them and the refit is
# an exact linear solve.

# Same as bias_terms(), in numpy.
def bias_vecs(params):
    return np.asarray(bias_terms(jnp.asarray(params)))

mats = [chi2_matrix(j) for j in range(phases)]
noise_idx = [a for a in (0, 1, 2, 3) if a in free]

cv_chi2 = np.zeros((phases, phases, 3))
for i in range(phases):
    params0 = best_params[i].copy()
    params0[:4] = 0
    v0 = bias_vecs(params0)
    # Column a of U: change in the bias vector per unit of noise parameter a.
    U = np.array([bias_vecs(np.where(np.arange(10) == a, 1.0, params0)) - v0 for a in noise_idx]).T
    for j in range(phases):
        M, nP = mats[j]
        v = v0 - U @ np.linalg.solve(U.T @ M @ U, U.T @ M @ v0)
        cv_chi2[i, j] = chi2_parts(v, M, nP)
assert np.allclose(cv_chi2[np.arange(phases), np.arange(phases)].sum(1), chi2.sum(1))  # diagonal reproduces the fits

np.save(write_root + 'CV_Chi2.npy', cv_chi2)

# Mean test chi2 per fit phase i (average over the 24 other phases j).
off = ~np.eye(phases, dtype=bool)
test = cv_chi2.sum(2)[off].reshape(phases, phases - 1).mean(1)
print(f'train chi2 = {chi2.sum(1).mean():.1f}   test chi2 = {test.mean():.1f}   '
      f'(test - train = {(test - chi2.sum(1)).mean():.1f})')

train chi2 = 3049.5   test chi2 = 3064.5   (test - train = 15.1)
